#

# Dimensional Model

In [0]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS marathos.gold")

In [0]:
# Läs in silver tabellen (OBT = cleaned dataset från silver_layer)
df = spark.table("marathos.silver.obt")

# Kontrollerar att alla kolumner finns i datasetet
display(df)
df.printSchema()

### Dim Event

In [0]:
# Skapa unik lista av events 
dim_event = df.select(
    "Event_name",
    "Event_distance_length"
).distinct()

# Skapa event_id
w_event = Window.orderBy("Event_name")

dim_event = dim_event.withColumn(
    "event_id",
    dense_rank().over(w_event))


# Visa resultat
display(dim_event)

In [0]:
# Spara resultatet i gold
dim_event.write.mode("overwrite").saveAsTable("marathos.gold.dim_event")

### Dim Athlete

In [0]:
# Skapa unik lista av idrottare 
dim_athlete = df.select(
    "Athlete_ID",
    "Athlete_country",
    "Athlete_gender",
    "Athlete_age_category",
    ).distinct()

w_athlete = Window.orderBy("Athlete_ID")

dim_athlete = dim_athlete.withColumn("athlete_id", dense_rank().over(w_athlete))

# Visa resultatet
display(dim_athlete)

In [0]:
# Spara resultatet i gold
dim_athlete.write.mode("overwrite").saveAsTable("marathos.gold.dim_athlete")

### FCT Results

In [0]:
from pyspark.sql.functions import dense_rank, col
from pyspark.sql.window import Window

spark.sql("DROP TABLE IF EXISTS marathos.gold.fct_results")

In [0]:

fct_results = df.join(dim_event, ["Event_name", "Event_distance_length"], "inner") \
    .join(dim_athlete, ["Athlete_ID", "Athlete_country", "Athlete_gender", "Athlete_age_category"], "inner") \
    .select(
        dim_event["event_id"],             
        dim_athlete["athlete_id"],         
        df["Event_dates"],
        col("Athlete_average_speed").cast("double").alias("Athlete_average_speed"),
        df["Athlete_performance"]
    )

# Skapa ett unikt result_id
w_result = Window.orderBy("event_id", "athlete_id")
fct_results = fct_results.withColumn("result_id", dense_rank().over(w_result))

# Spara till gold-lagret
fct_results.write.mode("overwrite").saveAsTable("marathos.gold.fct_results")

In [0]:
display(spark.table("marathos.gold.fct_results").limit(5))

# Views 

In [0]:
# Skapar en gold view med resultatdata för dashboard och analys

spark.sql("""CREATE OR REPLACE VIEW marathos.gold.vw_results AS SELECT
        f.result_id,
        f.event_id,
        f.Event_dates,
        e.Event_name,
        e.Event_distance_length,
        f.Athlete_performance,
        a.Athlete_country,
        a.Athlete_gender,
        a.Athlete_age_category,
        f.Athlete_average_speed,
        f.Athlete_id,
        a.Athlete_ID AS original_athlete_id
          
    FROM marathos.gold.fct_results f 
    JOIN marathos.gold.dim_event e 
          ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a 
          ON f.Athlete_id = a.Athlete_id
          
          """)

In [0]:
# Kontrollera att viewen fungerar

display(spark.table("marathos.gold.vw_results"))

In [0]:
# Skapar en gold view för distansbaserade lopp (km och mi)

spark.sql("""
CREATE OR REPLACE VIEW marathos.gold.vw_distance_events AS
SELECT *
FROM marathos.gold.vw_results
WHERE Event_distance_length LIKE '%km'
   OR Event_distance_length LIKE '%mi'
""")

In [0]:
display(spark.table("marathos.gold.vw_distance_events"))

In [0]:
# Skapar en gold view för tidsbaserade lopp (min och timmar)

spark.sql("""
CREATE OR REPLACE VIEW marathos.gold.vw_time_events AS
SELECT *
FROM marathos.gold.vw_results
WHERE Event_distance_length LIKE '%h'
""")
display(spark.table("marathos.gold.vw_time_events"))

In [0]:
# kontrollera att databasen och schema finns
spark.sql("USE CATALOG marathos")
spark.sql("USE SCHEMA gold")


In [0]:
# # Visar alla views i gold schema

display(spark.sql("SHOW VIEWS IN marathos.gold"))

# Dashboard

In [0]:
# Läser in gold view för dashboard
df_dashboard = spark.table("marathos.gold.vw_results")

# Visar data som ska användas i dashboarden
display(df_dashboard)

In [0]:
# Beräkna antal unika deltagare per land. 
athlete_per_country = spark.sql("""
SELECT
    Athlete_country,
    COUNT(DISTINCT Athlete_ID) AS athletes_per_country
FROM marathos.gold.vw_results
GROUP BY Athlete_country
ORDER BY athletes_per_country DESC
""")

display(athlete_per_country)

In [0]:
display(athlete_per_country)

In [0]:
df_dashboard.printSchema()

In [0]:
# Beräknar genomsnittlig hastighet per land
country_speed = spark.sql("""
SELECT
    Athlete_country,
    AVG(TRY_CAST(Athlete_average_speed AS DOUBLE)) AS avg_speed
FROM marathos.gold.vw_results
GROUP BY Athlete_country
ORDER BY avg_speed DESC
""")

display(country_speed)

In [0]:
# Beräknar hur många olika lopp som finns för varje distanstyp

events_by_type = spark.sql("""
SELECT
    Event_distance_length,
    COUNT(DISTINCT event_id) AS total_events
FROM marathos.gold.vw_results
GROUP BY Event_distance_length
ORDER BY total_events DESC
""")

display(events_by_type)

# Metrics


In [0]:
# Beräknar totalt antal unika deltagare
total_athletes = spark.sql("""
SELECT COUNT(DISTINCT Athlete_ID) AS total_athletes
FROM marathos.gold.vw_results
""")

display(total_athletes)

In [0]:
# Beräknar totalt antal lopp
total_events = spark.sql("""
SELECT COUNT(DISTINCT event_id) AS total_events
FROM marathos.gold.vw_results
""")

display(total_events)

In [0]:
# Beräknar totalt antal länder
total_countries = spark.sql("""
SELECT COUNT(DISTINCT Athlete_country) AS total_countries
FROM marathos.gold.vw_results
""")

display(total_countries)

##### Denna query används för att kontrollera att svaret från Genie är korrekt genom att räkna antal unika deltagare per land och hitta landet med flest deltagare.

In [0]:
spark.sql("""
SELECT
    Athlete_country,
    COUNT(DISTINCT Athlete_ID) AS athletes_per_country
FROM marathos.gold.vw_results
GROUP BY Athlete_country
ORDER BY athletes_per_country DESC
LIMIT 1
""").display()

In [0]:
%sql
DESCRIBE marathos.gold.vw_results;